# Two-Stage Pill Identification - Evaluation

**Prerequisites:** `train.ipynb` must have been run first so that:
- `PROJECT_DIR/prototype1/weights/best.pt` exists (YOLO weights)
- `pill_classifier_resnet50_best.pth` exists in `BASE_DIR` (ResNet-50 weights)
- `CROPPED_DIR/valid/` and `CROPPED_DIR/test/` exist (cropped pills)

This notebook covers:
1. YOLOv8 evaluation on the test set
2. ResNet-50 evaluation - confusion matrix & classification report
3. End-to-end inference on a single image

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q ultralytics torch torchvision scikit-learn pandas opencv-python matplotlib seaborn

In [ ]:
# Imports
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from ultralytics import YOLO


In [ ]:
# Paths — must match what was used in train.ipynb
BASE_DIR    = '/content/drive/My Drive/dl proj/Pill_identification'
DATASET_DIR = f'{BASE_DIR}/prototype_dataset'
PROJECT_DIR = f'{BASE_DIR}/outputs_results'
CROPPED_DIR = f'{BASE_DIR}/cropped_pills'
CSV_PATH    = f'{BASE_DIR}/pill_detailed_contents.csv'
YAML_PATH   = f'{DATASET_DIR}/data.yaml'

YOLO_WEIGHTS       = f'{PROJECT_DIR}/prototype1/weights/best.pt'
CLASSIFIER_WEIGHTS = os.path.join(BASE_DIR, 'pill_classifier_resnet50_best.pth')

print('Paths set.')


## 2. Stage 1 Evaluation - YOLOv8 on Test Set

Runs `model.val()` on the held-out test split and prints per-class mAP, precision, and recall.

In [ ]:
# Load the trained YOLO model
yolo = YOLO(YOLO_WEIGHTS)

# Evaluate on test split
results_test = yolo.val(data=YAML_PATH, split='test')
print(results_test)


## 3. Stage 1 - Visual Check on Test Images

Runs YOLO inference and saves annotated images so you can visually inspect detections.

In [ ]:
# Run inference and save annotated images to /content/runs/detect/predict
yolo.predict(
    source=f'{DATASET_DIR}/test/images',
    save=True,
    conf=0.5
)


## 4. Stage 2 Evaluation - Load ResNet-50 Classifier

Rebuilds the same model architecture used in training and loads the saved best weights.

In [ ]:
# Inference-time transform (same normalisation as training)
TRANSFORMS_VALID = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Derive class names from the cropped_pills/train folder structure
train_dataset = datasets.ImageFolder(os.path.join(CROPPED_DIR, 'train'), TRANSFORMS_VALID)
class_names   = train_dataset.classes
num_classes   = len(class_names)
print('Classes:', class_names)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Rebuild model architecture
classifier = models.resnet50(pretrained=False)
classifier.fc = nn.Sequential(
    nn.Linear(classifier.fc.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, num_classes)
)

# Load saved weights
classifier.load_state_dict(torch.load(CLASSIFIER_WEIGHTS, map_location=device))
classifier = classifier.to(device)
classifier.eval()
print('Classifier loaded.')


## 5. Stage 2 Evaluation - Confusion Matrix & Classification Report

Runs the classifier over the validation split and computes:  
- Raw confusion matrix  
- Normalised confusion matrix  
- Per-class precision / recall / F1

In [ ]:
# Run classifier on the validation set
valid_dataset = datasets.ImageFolder(os.path.join(CROPPED_DIR, 'valid'), TRANSFORMS_VALID)
valid_loader  = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=2)

all_labels = []
all_preds  = []

with torch.no_grad():
    for imgs, labels in valid_loader:
        imgs    = imgs.to(device)
        labels  = labels.to(device)
        outputs = classifier(imgs)
        preds   = outputs.argmax(dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)


In [ ]:
# Raw confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Raw)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Normalised confusion matrix
cm_norm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Normalized)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
# Per-class precision, recall, F1
print(classification_report(all_labels, all_preds, target_names=class_names))


## 6. End-to-End Inference - Single Image

Runs the full two-stage pipeline on one test image:  
YOLO detects and crops the pill → ResNet-50 classifies it → CSV lookup prints pill contents.

In [ ]:
# Load pill contents from CSV
pill_df       = pd.read_csv(CSV_PATH)
pill_contents = {
    row['pill_name']: f"{row['active_ingredient']} | {row['excipients']}"
    for _, row in pill_df.iterrows()
}

# ── Change this path to any test image you want to run on ──
test_img = f"{DATASET_DIR}/test/images/20210729_182940_jpg.rf.6e894f1a567bf75e867eefbf57ec1876.jpg"

results = yolo.predict(source=test_img, conf=0.5)

for r in results:
    img = r.orig_img.copy()

    for i, (box, _) in enumerate(zip(r.boxes.xyxy, r.boxes.cls)):
        x1, y1, x2, y2 = map(int, box)
        crop = img[y1:y2, x1:x2]

        # Convert crop to tensor and classify
        crop_pil    = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        crop_tensor = TRANSFORMS_VALID(crop_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            out      = classifier(crop_tensor)
            probs    = F.softmax(out, dim=1)
            conf, pred = torch.max(probs, 1)

        pred_name = class_names[pred.item()]
        details   = pill_contents.get(pred_name, 'Unknown')

        print(f'Pill {i+1}: {pred_name} | Confidence: {conf.item():.2f}')
        print(f'Contents: {details}\n')
